<a href="https://colab.research.google.com/github/ismethakanaydogmus/News_Classification/blob/main/News_Classification_V3_DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 News Classification V3.0: Deep Learning (BERTurk)

Bu aşamada, klasik makine öğrenmesi sınırlarını aşarak **Transformers** mimarisine geçiş yapıyoruz. Hedefimiz, kelimelerin köklerinden ziyade cümle içindeki anlamsal bağlamlarını (context) öğrenmek.

In this stage, we are moving beyond classical machine learning to the **Transformers** architecture. Our goal is to learn the semantic context of words rather than just their roots.

In [ ]:
# 1. Projeyi klonla ve dizine gir
!git clone https://github.com/ismethakanaydogmus/News_Classification.git
%cd News_Classification

# 2. V3 için gerekli "Hugging Face" kütüphanelerini kur
!pip install transformers[torch] datasets accelerate -U

import pandas as pd
import torch
from datasets import Dataset

# 3. GPU Kontrolü (V3.0 için hayati önem taşır!)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Model şu cihazda çalışacak: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## 🛠️ 1. Veri Hazırlama ve Etiketleme / Data Preparation

BERT modeli kategorileri (Ekonomi, Siyaset vb.) metin olarak değil, sayısal ID'ler olarak bekler. Bu aşamada:
1. Kategorileri benzersiz ID'lere eşleyeceğiz (`label2id`).
2. Veriyi Pandas formatından Hugging Face'in optimize edilmiş `Dataset` formatına çevireceğiz.

---

BERT expects categories as numerical IDs. We will map categories to unique IDs and convert the Pandas DataFrame into the highly optimized Hugging Face `Dataset` format.

In [ ]:
import pandas as pd
from datasets import Dataset

# 1. Veriyi yükle
df = pd.read_csv("master_news_dataset_v2.csv")

# 2. Kategori -> ID eşleşmesi (Label Mapping)
categories = sorted(df['category'].unique())
label2id = {label: i for i, label in enumerate(categories)}
id2label = {i: label for i, label in enumerate(categories)}

print(f"✅ Kategoriler ve ID karşılıkları:\n{label2id}")

# DataFrame'e sayısal etiketleri ekleyelim
df['label'] = df['category'].map(label2id)

# 3. Hugging Face Dataset formatına çevirme
# Sadece gerekli olan başlık ve etiket sütunlarını alıyoruz
raw_dataset = Dataset.from_pandas(df[['headline', 'label']])

# Eğitim (%80) ve Test (%20) olarak bölelim
dataset_dict = raw_dataset.train_test_split(test_size=0.2, seed=42)

print("\n✅ Veri seti bölündü:")
print(dataset_dict)

## 🔠 2. Tokenization: BERTurk ile Metin İşleme

BERT, kelimeleri doğrudan okumak yerine onları **Token** adı verilen parçalara böler. Bizim kullanacağımız **dbmdz/bert-base-turkish-cased** modeli, Türkçe'nin eklemeli yapısına (agglutinative) çok uygun olan **WordPiece** yöntemini kullanır.

* **Neden Önemli?** Bu yöntem sayesinde model "Gidemedim" kelimesini `["git", "##emedim"]` gibi parçalayarak kelimenin kökündeki eylemi ve ekin kattığı olumsuzluk/zaman anlamını ayrı ayrı kavrar.

---

BERT breaks down text into **Tokens** using the **WordPiece** method, which is ideal for Turkish. This allows the model to understand the semantic meaning of suffixes and complex word structures.

In [ ]:
from transformers import AutoTokenizer

# Türkçe BERT modelinin tokenizer'ını yükleyelim
model_checkpoint = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # padding: Cümleleri 64 karakter boyuna tamamlar
    # truncation: 64 karakterden uzun olanları keser
    return tokenizer(examples["headline"], padding="max_length", truncation=True, max_length=64)

# Tüm veri setini tokenize edelim (Map fonksiyonu çok hızlıdır)
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Orijinal metin sütununa artık ihtiyacımız yok, sayısal versiyonlar yeterli
tokenized_datasets = tokenized_datasets.remove_columns(["headline"])

print("✅ Tokenization tamamlandı! Model için hazırlanan örnek veri yapısı:")
print(tokenized_datasets["train"][0].keys())

## 🧠 3. Model Hazırlığı ve İnce Ayar / Model & Fine-Tuning

Şimdi, milyonlarca Türkçe cümle ile önceden eğitilmiş olan **BERTurk** modelini çağıracağız. Ancak bir farkla: Modelin en üst katmanını (head) söküp, yerine bizim 10 kategorimizi (Ekonomi, Astroloji vb.) sınıflandıracak yeni bir katman takacağız.

### ⚙️ Hiper-parametreler / Hyper-parameters
* **Learning Rate (2e-5):** Derin öğrenmede "öğrenme hızı" çok kritiktir. Çok yüksek olursa model şaşırır, çok düşük olursa yerinde sayar. 2e-5, BERT için altın orandır.
* **Epoch (3):** Tüm veri setini modelin önüne 3 kez getireceğiz. Fazlası "ezberleme" (overfitting) riskine yol açabilir.
* **Batch Size (16):** Verileri 16'şarlı gruplar halinde modele vereceğiz.

---

We will initialize the pre-trained **BERTurk** model and replace its classification head to match our 10 categories. We will use a learning rate of **2e-5** and train for **3 epochs**, which is the standard recipe for stable fine-tuning.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Modeli yükleyelim
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(categories), # 10 kategori
    id2label=id2label,
    label2id=label2id
).to(device)

# 2. Başarı metriklerini tanımlayalım
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

print(f"✅ {model_checkpoint} modeli {len(categories)} sınıf için hazırlandı!")

In [ ]:
from transformers import Trainer

# Trainer'ı oluştur - Transformers v5+ Uyumlu
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,       # 'tokenizer' yerine 'processing_class' yazıyoruz
    compute_metrics=compute_metrics,
)

# 🏁 EĞİTİMİ BAŞLAT / START TRAINING
print("🔥 V3.0 Fine-tuning (BERT) başlatıldı. T4 GPU devrede...")
trainer.train()

## 📊 4. V3.0 Performans Değerlendirmesi / Performance Evaluation

V3.0 (BERTurk) modelimiz, klasik yöntemlerin (LinearSVC) tıkandığı noktada bayrağı devraldı ve başarıyı **%82** seviyesine taşıdı.

### 🌟 Ana Gözlemler / Key Observations
* **İstikrarlı Öğrenme:** Training Loss her epoch'ta (0.74 -> 0.51 -> 0.38) düzenli olarak düştü. Bu, modelin ezberlemediğini, gerçekten öğrendiğini gösteriyor.
* **Bağlamsal Zeka:** Zeyrek gibi manuel kök bulma araçları kullanmamıza gerek kalmadan model, kelime eklerinden anlam çıkarmayı başardı.
* **Genelleme Yeteneği:** Validation Accuracy (**%82.3**), modelin hiç görmediği haber başlıklarını bile çok yüksek bir isabetle bildiğini kanıtlıyor.

---

V3.0 (BERTurk) outperformed classical methods by pushing accuracy to **82.3%**. The steady decrease in Training Loss (0.74 to 0.38) indicates healthy learning. The model successfully leveraged contextual understanding without the need for manual lemmatization.

In [ ]:
from sklearn.metrics import classification_report

# Test seti üzerinde tahminleri alalım
predictions = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions.predictions, axis=-1)

print("\n--- V3.0 BERTurk Model Detaylı Performans Raporu ---")
print(classification_report(tokenized_datasets["test"]["label"], preds, target_names=categories))

In [ ]:
def predict_v3(text):
    # Metni tokenize et
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)

    # Tahmin yap
    with torch.no_grad():
        outputs = model(**inputs)

    # En yüksek skora sahip etiketi bul
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(probs).item()

    return id2label[pred_idx]

# Canlı Test
test_headlines = [
    "Merkür retrosu bu hafta tüm burçları derinden etkileyecek",
    "Şampiyonlar Ligi finalinde dev randevu: Real Madrid vs City",
    "Yapay zeka modelleri yazılım dünyasında yeni bir devir açıyor",
    "Yeni bulunan antik kent arkeoloji dünyasını heyecanlandırdı",
    "Okullarda yeni müfredat çalışmaları tamamlanmak üzere",
    "Mars'ta su izine rastlanması bilim insanlarını şaşırttı",
    "Keskin kararlar zamanı",
    "Motorine bu gece indirim geliyor! İşte yeni fiyatlar...",
]

print(f"{'HABER BAŞLIĞI':<60} | {'V3.0 TAHMİN'}")
print("-" * 80)
for h in test_headlines:
    print(f"{h[:58]:<60} | {predict_v3(h)}")

In [ ]:
# Modeli ve Tokenizer'ı yerel klasöre kaydedelim
model.save_pretrained("./my_best_bert_model")
tokenizer.save_pretrained("./my_best_bert_model")

print("✅ V3.0 Modeli 'my_best_bert_model' klasörüne kaydedildi!")